# HW2: Deep Q-Network
All implementation lives in `homework2.py`. This notebook imports from it, runs training, and visualises results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from homework2 import (
    ReplayBuffer, DQNNetwork, DQNAgent, train,
    DEVICE, N_ACTIONS, STATE_DIM,
    MEMORY_SIZE, NUM_EPISODES, BATCH_SIZE,
    EPS_DECAY, EPS_END, EPS_START,
    GAMMA, LEARNING_RATE, TAU,
)

print(f"Using device: {DEVICE}")
print(f"episodes={NUM_EPISODES}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, tau={TAU}, gamma={GAMMA}")


In [ ]:
# Smoke tests
buf = ReplayBuffer(100)
buf.push(np.zeros(6), 0, 1.0, np.ones(6), False)
assert len(buf) == 1
s, a, r, ns, d = buf.sample(1)
assert s.shape == (1, 6)
print("ReplayBuffer OK")

net = DQNNetwork(STATE_DIM, N_ACTIONS).to(DEVICE)
dummy = torch.zeros(1, STATE_DIM).to(DEVICE)
assert net(dummy).shape == (1, N_ACTIONS)
print("DQNNetwork OK")

agent_test = DQNAgent()
action = agent_test.select_action(np.zeros(6))
assert 0 <= action < N_ACTIONS
print("DQNAgent OK")


In [ ]:
# Run training — resumes automatically if a checkpoint exists for this run_name
RUN_NAME = "run2"

agent, episode_rewards, episode_rps = train(
    run_name=RUN_NAME,
    n_splits=15,
    checkpoint_every=500,
)


In [ ]:
def smooth(data, window=50):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.3, color='steelblue', label='Raw')
axes[0].plot(range(49, len(episode_rewards)), smooth(episode_rewards), color='steelblue', label='Smoothed (50)')
axes[0].set_title('Episode Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()

axes[1].plot(episode_rps, alpha=0.3, color='darkorange', label='Raw')
axes[1].plot(range(49, len(episode_rps)), smooth(episode_rps), color='darkorange', label='Smoothed (50)')
axes[1].set_title('Reward Per Step (RPS)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('RPS')
axes[1].legend()

plt.tight_layout()
out_path = f'hw2_{RUN_NAME}_results.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f"Saved {out_path}")
